# SSB StatBank → Landing Layer
 
Henter data fra SSB PXWeb v2 API og lagrer til Landing (Bronze) layer.
## Endringer fra v6
- Leser tabeller fra `statbank_staging.pipeline.ssb_load_queue` og looper (ikke lenger én tabell manuelt)
- `get_time_dimension` har fallback via kjente navn og heuristikk
- `fetch_data_for_period` bruker faktisk dimensjonsnøkkel, ikke hardkodet `Tid`
- `should_refresh` parser datoer robust i stedet for streng-sammenligning
- `lookback_periods` fra køen styrer hvor mange perioder som refreshes
- `mark_table_as_downloaded` kalles etter vellykket ingest
## Kjøring 

Kjøres fra Fabric Pipeline etter `03_ssb_oppdateringsdetector`.

Kan også kjøres manuelt – leser da fra gjeldende `statbank_staging.pipeline.ssb_load_queue`.
  

In [ ]:
# =====================================================================
# PARAMETERE (overstyres av Fabric Pipeline)
# =====================================================================
# QUEUE_TABLE avgjør hvilke tabeller denne kjøringen henter data for – dette
# er køen 03_ssb_oppdateringsdetector nettopp har fylt. Peker vanligvis på
# "alle tabeller"-køen, men kan settes til kun-CRITICAL-køen for en
# raskere, prioritert kjøring.
# Sett QUEUE_TABLE til statbank_staging.pipeline.ssb_load_queue_critical for kritisk pipeline
QUEUE_TABLE   = "statbank_staging.pipeline.ssb_load_queue"   # eller statbank_staging.pipeline.ssb_load_queue_critical
CONFIG_TABLE  = "statbank_staging.pipeline.ssb_config"
LANG          = "no"
PXWEB_VERSION = "v2"

In [ ]:
# =====================================================================
# IMPORTS
# =====================================================================
# Standard oppsett – kobler til Spark og gjør biblioteker for HTTP-kall
# (requests) og Delta-tabeller tilgjengelig.
from __future__ import annotations
import json
import time
import datetime as dt
from datetime import timezone
from typing import Dict, List, Optional, Tuple
import requests
from requests.exceptions import RequestException
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable
spark = SparkSession.builder.appName("SSBIngestLanding").getOrCreate()
from notebookutils import mssparkutils
print("✅ Imports OK")


In [ ]:
# =====================================================================
# KONFIGURASJON
# =====================================================================
# Faste innstillinger: hvor i lakehouset dataene skal lagres (Files/landing/
# ssb/statbank/...), samt hvor tålmodig notebooken skal være mot SSB sitt
# API (timeout, antall forsøk, pauser mellom kall) for å unngå å bli
# rate-limitet.
LAKEHOUSE_ROOT = "Files"
ZONE_LANDING   = "landing"
DATA_SOURCE    = "ssb"
DATA_PRODUCT   = "statbank"
SSB_BASE       = "https://data.ssb.no/api/pxwebapi"
ORG_NAME       = "Akershus analyse"
HTTP_TIMEOUT   = 120
MAX_RETRIES    = 3
BACKOFF_BASE   = 1.0
BACKOFF_MAX    = 30.0
API_PAUSE      = 0.5   # sekunder mellom periode-kall
API_PAUSE_CHUNK = 0.05  # sekunder mellom chunk-kall
print("✅ Konfigurasjon OK")

In [ ]:
# =====================================================================
# FABRIC FILESYSTEM HELPERS
# =====================================================================
class FabricFS:
    """Abstraksjon for filoperasjoner via mssparkutils."""
    @staticmethod
    def normalize_path(path: str) -> str:
        if path.startswith("Files/"):
            return path
        if path.startswith("/lakehouse/default/Files/"):
            return path.replace("/lakehouse/default/", "")
        return path
    @staticmethod
    def exists(path: str) -> bool:
        path = FabricFS.normalize_path(path)
        try:
            mssparkutils.fs.ls(path)
            return True
        except Exception:
            return False
    @staticmethod
    def mkdirs(path: str) -> None:
        path = FabricFS.normalize_path(path)
        try:
            mssparkutils.fs.mkdirs(path)
        except Exception as e:
            if "already exists" not in str(e).lower():
                raise
    @staticmethod
    def write_json(path: str, data: dict, max_retries: int = 3) -> None:
        path = FabricFS.normalize_path(path)
        content = json.dumps(data, ensure_ascii=False, indent=2)
        for attempt in range(max_retries):
            try:
                mssparkutils.fs.put(path, content, overwrite=True)
                return
            except Exception as e:
                if attempt < max_retries - 1:
                    wait = 2 ** attempt
                    print(f"   Skrivefeil, forsøk {attempt + 1}/{max_retries}, venter {wait}s: {e}")
                    time.sleep(wait)
                else:
                    raise
    @staticmethod
    def read_json(path: str) -> dict:
        path = FabricFS.normalize_path(path)
        content = mssparkutils.fs.head(path, 10_000_000)
        return json.loads(content)
    @staticmethod
    def list_snapshot_dirs(landing_base: str) -> List[str]:
        """List snapshot-mapper sortert nyeste forst (returnerer mappenavn, ikke full sti)."""
        path = FabricFS.normalize_path(landing_base)
        try:
            items = mssparkutils.fs.ls(path)
            dirs = [item.name for item in items if item.isDir()]
        except Exception:
            return []
        return sorted([d for d in dirs if "snapshot_date=" in d], reverse=True)
print("✅ FabricFS OK")


In [ ]:
# =====================================================================
# HTTP CLIENT
# =====================================================================
# En generell hjelper for å gjøre HTTP-kall mot SSB med automatisk
# gjenforsøk – venter og prøver på nytt hvis SSB svarer "for mange
# forespørsler" (HTTP 429) eller ved forbigående nettverksfeil.
class HTTPClient:
    """HTTP GET med eksponentiell backoff og rate-limit-håndtering."""
    @staticmethod
    def get(
        url: str,
        table_id: str = "",
        timeout: int = HTTP_TIMEOUT,
    ) -> requests.Response:
        headers = {
            "User-Agent": f"{ORG_NAME}-StatBank-Ingest/7.0-fabric (table {table_id})",
            "Accept": "application/json",
            "Accept-Language": "no",
        }
        last_error: Optional[Exception] = None
        for attempt in range(MAX_RETRIES + 1):
            try:
                resp = requests.get(url, headers=headers, timeout=timeout)
                if resp.status_code == 429:
                    wait = min(BACKOFF_BASE * (2 ** attempt), BACKOFF_MAX)
                    print(f"   HTTP 429 – venter {wait:.1f}s...")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                return resp
            except RequestException as e:
                last_error = e
                if attempt < MAX_RETRIES:
                    wait = min(BACKOFF_BASE * (2 ** attempt), BACKOFF_MAX)
                    print(f"   Nettverksfeil, forsøk {attempt + 1}/{MAX_RETRIES}, venter {wait:.1f}s: {e}")
                    time.sleep(wait)
                else:
                    raise
        raise last_error
print("✅ HTTPClient OK")

In [ ]:
# =====================================================================
# SSB API-FUNKSJONER
# =====================================================================
# Funksjonene som faktisk snakker med SSB sitt API: hente grunnleggende
# info om en tabell, hente full struktur/metadata, finne ut hvilke
# tidsperioder som finnes, og til slutt hente selve tallene for én periode
# om gangen. get_time_dimension er verdt å merke seg – SSB kaller ikke
# tidsdimensjonen det samme i alle tabeller, så denne prøver flere
# strategier for å finne den uansett.
def fetch_table_info(table_id: str) -> dict:
    """Hent tabell-info – uten /no/ sti som gir 404."""
    url = f"{SSB_BASE}/{PXWEB_VERSION}/tables/{table_id}/?lang=no"
    resp = requests.get(
        url,
        headers={"Accept": "application/json", "Accept-Language": "no"},
        timeout=HTTP_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()


def fetch_metadata(table_id: str) -> dict:
    """Hent full JSON-stat2 metadata – uten /no/ sti."""
    url = f"{SSB_BASE}/{PXWEB_VERSION}/tables/{table_id}/data?lang=no&content=metadata"
    print(f"   Metadata-url: {url}") 
    resp = requests.get(
        url,
        headers={"Accept": "application/json", "Accept-Language": "no"},
        timeout=HTTP_TIMEOUT,
    )
    resp.raise_for_status()
    data = resp.json()
    if "dimension" not in data:
        raise RuntimeError(f"Mangler 'dimension' i metadata for {table_id}")
    return data

def fetch_v1_table_info(table_id: str) -> dict:
    """Hent dimensjoner og tidskoder fra PxWeb v1 i ett kall."""
    url = f"https://data.ssb.no/api/v0/no/table/{table_id}"
    resp = requests.get(url, headers={"Accept": "application/json"}, timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    dims = [var["code"] for var in data.get("variables", [])]
    periods = []
    for var in data.get("variables", []):
        if var["code"] in ("Tid", "tid", "Time", "time"):
            periods = var["values"]
            break
    return {"dims": dims, "periods": periods}


def get_time_dimension(metadata: dict) -> str:
    """
    Finn navn på tidsdimensjonen i metadata.
    Prøver i rekkefølge:
      1) role.time (JSON-Stat2 standard)
      2) Kjente SSB-navn (Tid, Time, tid, time)
      3) Heuristikk: dimensjonsnavn som inneholder tidsord
    """
    dims = metadata.get("dimension", {})
    # 1) JSON-Stat2 role.time
    role = metadata.get("role", {})
    time_keys = role.get("time", []) if isinstance(role, dict) else []
    if time_keys and time_keys[0] in dims:
        print(f"   Tidsdimensjon funnet via role.time: '{time_keys[0]}'")
        return time_keys[0]
    # 2) Kjente standardnavn
    for known in ("Tid", "tid", "Time", "time"):
        if known in dims:
            print(f"   Tidsdimensjon funnet via kjent navn: '{known}'")
            return known
    # 3) Heuristikk
    time_words = {"tid", "time", "aar", "år", "year", "periode",
                  "period", "kvartal", "quarter", "maaned", "month"}
    for key in dims:
        if key.lower() in time_words:
            print(f"   Tidsdimensjon funnet via heuristikk: '{key}'")
            return key
    raise ValueError(
        f"Fant ikke tidsdimensjon. Tilgjengelige dimensjoner: {list(dims.keys())}"
    )
def get_time_periods(metadata: dict, time_dim: str) -> List[str]:
    """Hent sorterte tidskoder fra metadata."""
    dim = metadata.get("dimension", {}).get(time_dim, {})
    cat = dim.get("category", {})
    # Prøv index (er en dict: kode -> posisjon)
    index = cat.get("index", {})
    if isinstance(index, dict) and index:
        codes = sorted(index.keys(), key=lambda k: index[k])
        return codes
    # Fallback: label-keys
    labels = cat.get("label", {})
    if isinstance(labels, dict) and labels:
        return list(labels.keys())
    raise ValueError(f"Fant ingen tidskoder i dimensjonen '{time_dim}'")

def fetch_data_for_period(
    table_id: str,
    time_dim: str,
    period: str,
    all_dims: List[str],
) -> dict:
    """Hent data for én periode – manuell URL-bygging for å unngå bracket-encoding."""
    dim_params = "&".join(
        f"valueCodes[{dim}]={period}" if dim == time_dim else f"valueCodes[{dim}]=*"
        for dim in all_dims
    )
    url = f"{SSB_BASE}/{PXWEB_VERSION}/tables/{table_id}/data?lang={LANG}&outputFormat=json-stat2&{dim_params}"
    print(f" Data-url [{period}]: {url}")

    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            req = requests.Request(
                "GET",
                url,
                headers={
                    "User-Agent": f"{ORG_NAME}-StatBank-Ingest/7.0-fabric (table {table_id})",
                    "Accept": "application/json",
                    "Accept-Language": "no",
                },
            )
            prep = req.prepare()
            prep.url = url  # tving rå URL, hindrer re-encoding av brackets
            resp = requests.Session().send(prep, timeout=HTTP_TIMEOUT)
            
            if resp.status_code == 429:
                time.sleep(min(BACKOFF_BASE * (2 ** attempt), BACKOFF_MAX))
                continue
            resp.raise_for_status()
            data = resp.json()
            if "value" not in data:
                raise ValueError(f"Mangler 'value' i respons for periode {period}")
            return data
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES:
                wait = min(BACKOFF_BASE * (2 ** attempt), BACKOFF_MAX)
                print(f"   Nettverksfeil, forsok {attempt + 1}/{MAX_RETRIES}, venter {wait:.1f}s: {e}")
                time.sleep(wait)
            else:
                raise
    raise last_error

def fetch_data_for_period_chunked(
    table_id: str,
    time_dim: str,
    period: str,
    all_dims: List[str],
    chunk_dim: str,
    chunk_values: List[str],
) -> dict:
    """Hent data i chunks langs chunk_dim, slå sammen til én respons."""
    combined_values = []
    combined_data = None

    for i, chunk_val in enumerate(chunk_values):
        dim_params = "&".join(
            f"valueCodes[{dim}]={period}" if dim == time_dim
            else f"valueCodes[{dim}]={chunk_val}" if dim == chunk_dim
            else f"valueCodes[{dim}]=*"
            for dim in all_dims
        )
        url = f"{SSB_BASE}/{PXWEB_VERSION}/tables/{table_id}/data?lang={LANG}&outputFormat=json-stat2&{dim_params}"
        print(f"   Chunk [{i+1}/{len(chunk_values)}]: {url}") 
        last_err = None
        for attempt in range(MAX_RETRIES + 1):
            try:
                session = requests.Session()
                session.headers.update({
                    "User-Agent": f"{ORG_NAME}-StatBank-Ingest/7.0-fabric (table {table_id})",
                    "Accept": "application/json",
                    "Accept-Language": "no",
                })
                prep = requests.Request("GET", url).prepare()
                prep.url = url
                resp = session.send(prep, timeout=HTTP_TIMEOUT)
                if resp.status_code == 429:
                    wait = min(BACKOFF_BASE * (2 ** attempt), BACKOFF_MAX)
                    print(f"   HTTP 429 – venter {wait:.1f}s...")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                data = resp.json()
                break  # vellykket – gå videre til neste chunk
            except Exception as e:
                last_err = e
                if attempt < MAX_RETRIES:
                    wait = min(BACKOFF_BASE * (2 ** attempt), BACKOFF_MAX)
                    print(f"   Chunk-feil {attempt+1}/{MAX_RETRIES}, venter {wait:.1f}s: {e}")
                    time.sleep(wait)
                else:
                    raise last_err

        if combined_data is None:
            combined_data = data
        else:
            combined_data["value"].extend(data.get("value", []))

        time.sleep(API_PAUSE_CHUNK)

    return combined_data

In [ ]:
# =====================================================================
# CHANGE DETECTION
# =====================================================================
# En ekstra sikkerhetssjekk før vi laster ned noe: har SSB faktisk publisert
# noe nytt siden forrige gang denne tabellen ble hentet? Sjekken gjøres ved
# å sammenligne mot manifestet fra forrige snapshot. 03 har allerede gjort
# en lignende vurdering før tabellen havnet i køen, men her fanges det opp
# hvis noe skulle ha endret seg i mellomtiden.
def _parse_dt(ts: Optional[str]) -> Optional[dt.datetime]:
    """Robust ISO-timestamp-parsing med timezone-normalisering til UTC."""
    if not ts:
        return None
    for fmt in ("%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%dT%H:%M%z"):
        try:
            d = dt.datetime.strptime(ts.replace("Z", "+00:00"), fmt)
            if d.tzinfo is None:
                d = d.replace(tzinfo=timezone.utc)
            return d
        except ValueError:
            continue
    return None
def get_latest_manifest(landing_base: str) -> Optional[dict]:
    """Finn og les siste manifest fra forrige snapshot."""
    snapshots = FabricFS.list_snapshot_dirs(landing_base)
    if not snapshots:
        return None
    for snap_name in snapshots:
        manifest_path = f"{landing_base}/{snap_name}/manifest.json"
        try:
            if FabricFS.exists(manifest_path):
                return FabricFS.read_json(manifest_path)
        except Exception:
            continue
    return None
def should_refresh(ssb_updated: Optional[str], landing_base: str) -> bool:
    """
    Sjekk om SSB har ny data siden siste ingest.
    Bruker datetime-sammenligning, ikke streng-sammenligning.
    """
    manifest = get_latest_manifest(landing_base)
    if manifest is None:
        print("   Ingen tidligere snapshot – kjorer first load")
        return True
    prev_updated = manifest.get("ssb_updated")
    ssb_dt  = _parse_dt(ssb_updated)
    prev_dt = _parse_dt(prev_updated)
    if ssb_dt is None:
        print("   Ingen 'updated' timestamp fra SSB – kjorer for sikkerhets skyld")
        return True
    if prev_dt is None or ssb_dt > prev_dt:
        print(f"   Ny data: {ssb_updated} > {prev_updated}")
        return True
    print(f"   Ingen endring siden siste snapshot ({ssb_updated})")
    return False
print("✅ Change detection OK")

In [ ]:
# =====================================================================
# MARK TABLE AS DOWNLOADED (oppdater ssb_config)
# =====================================================================
def _ensure_last_downloaded_timestamp_column() -> None:
    """
    Migrering: legg til last_downloaded_timestamp-kolonne i ssb_config
    hvis den mangler. Trygg å kalle gjentatte ganger. Speiler samme
    migrering som finnes i 02_ssb_config_admin/03_ssb_oppdateringsdetector,
    slik at 04 er robust selv om den kjøres manuelt uten at de har kjørt.
    """
    try:
        cols = [f.name for f in spark.table(CONFIG_TABLE).schema.fields]
        if "last_downloaded_timestamp" not in cols:
            print("⚙️  Legger til manglende 'last_downloaded_timestamp'-kolonne i ssb_config...")
            spark.sql(f"ALTER TABLE {CONFIG_TABLE} ADD COLUMN last_downloaded_timestamp STRING")
            print("✅ 'last_downloaded_timestamp'-kolonne lagt til")
    except Exception:
        pass  # tabellen finnes kanskje ikke ennå


def mark_table_as_downloaded(table_id: str) -> None:
    """
    Oppdater last_downloaded_timestamp i ssb_config etter vellykket
    nedlasting til landing. Skiller seg fra last_loaded_timestamp (nb 05)
    som settes først når data er standardisert til silver.
    """
    _ensure_last_downloaded_timestamp_column()
    timestamp = dt.datetime.now(timezone.utc).isoformat()
    try:
        DeltaTable.forName(spark, CONFIG_TABLE).update(
            condition=F.col("table_id") == table_id,
            set={"last_downloaded_timestamp": F.lit(timestamp)},
        )
        print(f"   ssb_config oppdatert: {table_id} → last_downloaded_timestamp {timestamp[:19]}")
    except Exception as e:
        print(f"   ADVARSEL: Kunne ikke oppdatere ssb_config for {table_id}: {e}")
print("✅ mark_table_as_downloaded OK")

In [ ]:
# =====================================================================
# INGEST – ÉN TABELL
# =====================================================================
# Selve hovedarbeidet for én tabell, i sju steg (merket i koden under):
# sjekk om noe er nytt, finn ut hvilke tidsperioder som finnes, last ned
# hver periode som en egen JSON-fil, og skriv til slutt et manifest som
# oppsummerer hva som ble hentet. Eldre perioder som allerede er lastet ned
# tidligere røres ikke – kun de siste periodene (styrt av lookback_periods)
# hentes på nytt hver gang, siden SSB av og til reviderer ferske tall.
def ingest_table(
    table_id: str,
    lookback_periods: int,
    snapshot_date: str,
    queue_check_timestamp: Optional[str] = None,
) -> dict:
    """
    Hent og lagre alle perioder for én SSB-tabell.
    Eldre perioder beholdes fra forrige snapshot (semi-inkrementell).
    Kun de siste `lookback_periods` periodene refreshes alltid.
    Returns:
        dict med status, perioder, observasjoner, feil
    """
    landing_base = f"{LAKEHOUSE_ROOT}/{ZONE_LANDING}/{DATA_SOURCE}/{DATA_PRODUCT}/{table_id}"
    snapshot_dir = f"{landing_base}/snapshot_date={snapshot_date}"
    result = {
        "status":          "unknown",
        "table_id":        table_id,
        "snapshot_date":   snapshot_date,
        "snapshot_dir":    snapshot_dir,
        "ssb_updated":     None,
        "table_label":     None,
        "periods_total":   0,
        "periods_saved":   0,
        "observations":    0,
        "bytes":           0,
        "error":           None,
    }
    print(f"\n{'='*60}")
    print(f"Tabell: {table_id}  |  lookback: {lookback_periods}")
    print(f"{'='*60}")
    try:
        # --- Steg 1: Tabell-info ---
        info = fetch_table_info(table_id)
        table_label = info.get("label", table_id)
        ssb_updated = info.get("updated")
        result["table_label"] = table_label
        result["ssb_updated"] = ssb_updated
        print(f"Label:       {table_label}")
        print(f"SSB updated: {ssb_updated}")
        # --- Steg 2: Change detection ---
        if not should_refresh(ssb_updated, landing_base):
            result["status"] = "skipped"
            return result
      
        # --- Steg 3: Metadata, dimensjoner og perioder ---
        metadata = fetch_metadata(table_id)
        time_dim = get_time_dimension(metadata)

        v1 = fetch_v1_table_info(table_id)

        all_dims = v1["dims"] or list(metadata.get("dimension", {}).keys())
        print(f"   Dimensjoner: {all_dims}")

        periods = get_time_periods(metadata, time_dim)
        if len(periods) <= 1 and v1["periods"]:
            print("   ⚠️ v2 returnerte kun 1 periode – bruker v1 fallback")
            periods = v1["periods"]

        result["periods_total"] = len(periods)
        print(f"Tidsdim:     {time_dim}")
        print(f"Perioder:    {len(periods)} ({periods[0]} → {periods[-1]})")
        print(f"Refresh:     siste {lookback_periods} perioder")

        # --- Steg 4: Opprett snapshot-mappe ---
        FabricFS.mkdirs(snapshot_dir)

        # --- Steg 5: Last ned perioder ---
        total_obs   = 0
        total_bytes = 0
        periods_saved = 0
        for idx, period in enumerate(periods, 1):
            data_path = f"{snapshot_dir}/period={period}/ssb_{table_id}_{period}.json"
            # Behold eksisterende data for historiske perioder
            position_from_end = len(periods) - idx + 1
            if position_from_end > lookback_periods and FabricFS.exists(data_path):
                print(f"   [{idx}/{len(periods)}] {period}: beholder eksisterende")
                periods_saved += 1
                continue
            try:
                data = fetch_data_for_period(table_id, time_dim, period, all_dims)
                n_obs = len([v for v in data.get("value", []) if v is not None])
                n_bytes = len(json.dumps(data).encode("utf-8"))
                FabricFS.mkdirs(f"{snapshot_dir}/period={period}")
                FabricFS.write_json(data_path, data)
                total_obs += n_obs
                total_bytes += n_bytes
                periods_saved += 1
                print(f"   [{idx}/{len(periods)}] {period}: {n_obs:,} obs, {n_bytes:,} bytes")
                if idx < len(periods):
                    time.sleep(API_PAUSE)
            except Exception as e:
                if "Too many cells" in str(e) or "400 Client Error" in str(e):
                    print(f"   ⚠️ For mange celler – prøver chunking på første dimensjon")
                    try:
                        chunk_dim = all_dims[0]
                        v1_data = requests.get(
                            f"https://data.ssb.no/api/v0/no/table/{table_id}",
                            headers={"Accept": "application/json"}
                        ).json()
                        chunk_values = next(
                            (var["values"] for var in v1_data["variables"] if var["code"] == chunk_dim),
                            []  # ← default tom liste i stedet for StopIteration
                        )
                        if not chunk_values:
                            print(f"   Fant ingen chunk-verdier for {chunk_dim} – skipper {period}")
                            result["error"] = f"Ingen chunk-verdier for {chunk_dim}"
                            continue
                        data = fetch_data_for_period_chunked(
                            table_id, time_dim, period, all_dims, chunk_dim, chunk_values
                        )
                        n_obs = len([v for v in data.get("value", []) if v is not None])
                        n_bytes = len(json.dumps(data).encode("utf-8"))
                        FabricFS.mkdirs(f"{snapshot_dir}/period={period}")
                        FabricFS.write_json(data_path, data)
                        total_obs += n_obs
                        total_bytes += n_bytes
                        periods_saved += 1
                        print(f"   [{idx}/{len(periods)}] {period}: {n_obs:,} obs, {n_bytes:,} bytes (chunked)")
                        if idx < len(periods):
                            time.sleep(API_PAUSE)
                    except Exception as chunk_e:
                        print(f"   [{idx}/{len(periods)}] {period}: FEIL (chunking) – {chunk_e}")
                        result["error"] = str(chunk_e)
                        continue
                else:
                    print(f"   [{idx}/{len(periods)}] {period}: FEIL – {e}")
                    result["error"] = str(e)
                    continue


        result["periods_saved"] = periods_saved
        result["observations"]  = total_obs
        result["bytes"]         = total_bytes
        # --- Steg 6: Manifest ---
        manifest = {
            "table_id":       table_id,
            "table_label":    table_label,
            "snapshot_date":  snapshot_date,
            "ssb_updated":    ssb_updated,
            "lang":           LANG,
            "pxweb_version":  PXWEB_VERSION,
            "time_dimension": time_dim,
            "periods_total":  len(periods),
            "periods_saved":  periods_saved,
            "observations":   total_obs,
            "bytes":          total_bytes,
            "lookback_periods": lookback_periods,
            "created_utc":    dt.datetime.now(timezone.utc).isoformat(),
            # check_timestamp fra ssb_load_queue-raden som utløste denne ingesten –
            # lar 05 oppdage om køen har blitt overskrevet av en ny 03-kjøring
            # mens 04/05 fortsatt jobbet med denne snapshoten.
            "queue_check_timestamp": queue_check_timestamp,
        }
        FabricFS.write_json(f"{snapshot_dir}/manifest.json", manifest)
        FabricFS.write_json(f"{snapshot_dir}/_SUCCESS", {"completed_utc": manifest["created_utc"]})
        print(f"\nFerdig: {periods_saved}/{len(periods)} perioder, {total_obs:,} obs, {total_bytes/1024/1024:.1f} MB")
        result["status"] = "success"
        return result
    except Exception as e:
        print(f"\nFEIL: {e}")
        import traceback
        traceback.print_exc()
        result["status"] = "failed"
        result["error"]  = str(e)
        return result
print("✅ ingest_table OK")

In [ ]:

# =====================================================================
# HOVEDLOOP – les fra kø og ingest alle tabeller
# =====================================================================
# Går gjennom hver tabell i køen og kaller ingest_table for den. Etter en
# vellykket nedlasting merkes tabellen som lastet ned i ssb_config (slik at
# 03 vet det neste gang den kjører), og til slutt skrives et sammendrag av
# hvor mange som gikk bra, ble hoppet over, eller feilet.
snapshot_date = dt.datetime.now(timezone.utc).strftime("%Y-%m-%d")
print(f"\nLaster fra ko: {QUEUE_TABLE}")
print(f"Snapshot-dato: {snapshot_date}")
print("="*60)
# Les tabeller fra køen
if not spark.catalog.tableExists(QUEUE_TABLE):
    raise RuntimeError(
        f"Ko-tabell '{QUEUE_TABLE}' finnes ikke. "
        "Kjor 03_ssb_oppdateringsdetector forst."
    )
queue_rows = spark.table(QUEUE_TABLE).collect()
if not queue_rows:
    print("Ko er tom – ingen tabeller a laste. Avslutter.")
else:
    print(f"Fant {len(queue_rows)} tabeller i koen\n")
    results     = []
    ok_count    = 0
    skip_count  = 0
    fail_count  = 0
    for row in queue_rows:
        table_id        = row["table_id"]
        lookback        = int(row["lookback_periods"]) if row["lookback_periods"] else 2
        result = ingest_table(
            table_id=table_id,
            lookback_periods=lookback,
            snapshot_date=snapshot_date,
            queue_check_timestamp=getattr(row, "check_timestamp", None),
        )
        results.append(result)
        if result["status"] == "success":
            mark_table_as_downloaded(table_id)
            ok_count += 1
        elif result["status"] == "skipped":
            skip_count += 1
        else:
            fail_count += 1
    # Sammendrag
    print(f"\n{'='*60}")
    print(f"SAMMENDRAG")
    print(f"{'='*60}")
    print(f"  Vellykket: {ok_count}")
    print(f"  Hoppet over (ingen endring): {skip_count}")
    print(f"  Feilet:    {fail_count}")
    print()
    for r in results:
        status_icon = {"success": "OK", "skipped": "--", "failed": "!!", "unknown": "??"}.get(r["status"], "?")
        print(
            f"  [{status_icon}] {r['table_id']}: "
            f"{str(r.get('table_label', ''))[:35]:<35} "
            f"{r['periods_saved']}/{r['periods_total']} perioder, "
            f"{r['observations']:,} obs"
        )
    if fail_count > 0:
        print("\nFeil-detaljer:")
        for r in results:
            if r["status"] == "failed":
                print(f"  {r['table_id']}: {r['error']}")
    print(f"\n{'='*60}")
    print("INGEST FULLFORT")
    print(f"{'='*60}")    